In [5]:
import numpy as np
from typing import Any
from lf_toolkit.evaluation import Result, Params

In [6]:
import os
import json

cwd = os.getcwd()
dir = os.path.dirname(cwd)
reference_path = os.path.join(dir, "data", "referenceMIDI.json")
response_path = os.path.join(dir, "data", "responseMIDI.json")

with open(reference_path) as f1:
    reference = json.load(f1)

with open(response_path) as f2:
    response = json.load(f2)

# Note Alignment techniques

## DTW

The goal of the note alignment is to find if the student played any missing or extra notes, and which note is missing/extra.


Based on the findings during the project plan phase, DTW is commonly used for alignment. The algorithm can be found in this book (Chpater 3.2 Dynamic Time Warping):
M. Müller, Fundamentals of Music Processing. Cham: Springer International Publishing, 2021, ISBN: 9783030698072. DOI:https://doi.org/10.1007/978-3-030-69808-9.

In general, this DTW algorithm finds an optimal possibly nonlinear alignment between response MIDI sequence to reference MIDI sequence.

Basic approach:
- Evaluating the local cost measure for each pair of elements in the response(X) and reference(Y) sequences. 
- Dynamic programming to find an alignment path between X and Y having minimal overall cost, i.e. DTW distance. The algorithm computes a cumulative distance path, the timestamps of the target MIDI are warped so they perfectly align with the anchor points of the reference MIDI.


However, this basic approach will not correctly handle the missing note case as expected, because it allows a note to match with multiple notes. Let's say, there is a note missing in the response, this algorithm tends to match a response note with two reference note, instead of reporting the missing problem.

! need to modify this algorithm to make it handle the missing/extra problem correctly. Cosider analysing each entry of the warping path.

In [ ]:
def compute_cost(note1, note2):
    """
    Compute the local cost measure for each pair of notes.
 
    Only pitch is involved in the cost calculation, since the purpose 
    is to pair up notes with similar pitches.
 
    Args:
        note1: dict with keys "pitch" (int), "start" (float), "duration" (float)
        note2: dict with keys "pitch" (int), "start" (float), "duration" (float)
 
    Returns:
        int: cost value >= 0 (lower means more similar pitch)
    """
    return int(abs(note1["pitch"] - note2["pitch"]))


def note_alignment_DTW(response_notes, ref_notes):
    """
    DTW pipeline: build cost matrix C -> build accumulated cost matrix D 
    -> backtrack to find the optimal warping path
    - the Rows of C and D correspond to response notes
    - the Columns of C and D correspond to reference notes
    
    Args:
        response_notes: The student's response MIDI notes to evaluate
        ref_notes: The reference MIDI note
 
    Returns:
        path: list of (response_idx, ref_idx) pairs — the optimal alignment
        C: local cost matrix
        D: accumulated cost matrix
    """

    N = len(response_notes)
    M = len(ref_notes)

    # step1: Build the local cost matrix C of size (N x M).
    # C[i, j] = note_cost(ref_notes[i], response_notes[j])
    C= np.zeros((N, M))
    for i in range(N):
        for j in range(M):
            C[i, j] = compute_cost(response_notes[i], ref_notes[j])

    # step2: Build the accumulated cost matrix D of size (N+1 x M+1)
    # using small trick for simplifying the initialization
    # D[n, 0] = inf  for n >= 1
    # D[0, m] = inf  for m >= 1
    D = np.full((N + 1, M + 1), np.inf)
    D[0, 0] = 0
    # for all n in [1..N] and m in [1..M]:
    # D[i, j] = C[i, j] + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            D[i, j] = C[i - 1, j - 1] + min(
                D[i - 1, j], # vertical step, multiple response notes are mapped to the same ref note
                D[i, j - 1], # horizontal step, same response note is reused for multiple ref notes
                D[i - 1, j - 1]) # diagonal step

    # step3: Backtrack through the D to find the optimal warping path P
    path = []
    n, m = N, M
    while n > 0 and m > 0:
        path.append((n-1, m-1))
        # Find the minimum cost step
        diag = D[n-1, m-1]
        vertical = D[n-1, m]
        horizontal = D[n, m-1]
        min_step = min(diag, vertical, horizontal)
        if min_step == vertical:
            n -= 1
        elif min_step == horizontal:
            m -= 1
        else: # diagonal step
            n -= 1
            m -= 1
    # Reverse to get the path from start to end
    path.reverse()  

    return path, C, D


def path_classification(path, C):
    """
    Classify the each entry of the alignment path into:
    - correct: response pitch matches reference pitch (cost = 0)
    - wrong: response pitch differs from reference pitch, but all the parings are one-to-one (cost > 0)
    - missing: same response pitch is mapped to multiple reference pitches 
    - extra: multiple response pitches are mapped to the same reference pitch
 
    Args:   
    path: list of (response_idx, reference_idx) pairs from the backtrack warping path
    C: local cost matrix (N x M)

    Returns:
        list of event dicts, each one of:
            {'type': 'match' or 'replacement' or 'missing' or 'extra', 
            'response_idx': int, 'ref_idx': int, 'cost': int}
    """

Evaluation

In [ ]:
def evaluate_note_pair(response_note, ref_note, ref_idx,
                       timing_tolerance=0.1, duration_tolerance=0.1):
    """
    Evaluate a single aligned note pair and return feedback.
 
    Args:
        response_note: student's note dict
        ref_note: reference note dict
        ref_idx: 1-based display index (based on ref position)
        timing_tolerance: consider as correct if start is within this tolerance
        duration_tolerance: consider as correct if duration is within this tolerance
 
    Returns:
        is_correct (bool), feedback (list of str)
    """
    feedback = []
    is_correct = True
 
    # Pitch check
    if response_note["pitch"] != ref_note["pitch"]:
        is_correct = False
        feedback.append(
            f"Note {ref_idx}: wrong pitch — expected {ref_note['pitch']}, "
            f"played {response_note['pitch']}."
        )
 
    # Timing check
    timing_diff = abs(response_note["start"] - ref_note["start"])
    if timing_diff > timing_tolerance:
        is_correct = False
        feedback.append(f"Note {ref_idx}: difference in start time: {timing_diff:.2f}s.")
 
    # Duration check
    duration_diff = abs(response_note["duration"] - ref_note["duration"])
    if duration_diff > duration_tolerance:
        is_correct = False
        feedback.append(f"Note {ref_idx}: difference in duration: {duration_diff:.2f}s.")
 
    if is_correct:
        feedback.append(f"Note {ref_idx} (with pitch {ref_note['pitch']}) is correct.")
 
    return is_correct, feedback

In [ ]:
def comparison(response, ref,
                   timing_tolerance=0.1, duration_tolerance=0.1):
    """
    Compare student MIDI against reference MIDI after DTW-based alignment.
 
    Args:
        response: The student's response MIDI
        ref: The reference MIDI
        timing_tolerance:   seconds
        duration_tolerance: seconds
 
    Returns:
        all_correct (bool), feedback (list of str)
    """
    response_notes = response["notes"]
    ref_notes = ref["notes"]
 
    # Align using DTW — response first, ref second
    path, C, D = note_alignment_DTW(response_notes, ref_notes)
 
    feedback = []
    all_correct = True
    
    for response_idx, ref_idx in path:
        is_correct, feedback = evaluate_note_pair(
            response_notes[response_idx], ref_notes[ref_idx],
            ref_idx=ref_idx + 1,
            timing_tolerance=timing_tolerance,
            duration_tolerance=duration_tolerance,
        )
        if not is_correct:
            all_correct = False
        feedback.extend(feedback)
 
    return all_correct, feedback

In [ ]:
def evaluation_function(response: Any, answer: Any, params: Params) -> Result:
    """
    Entry point for Lambda Feedback.
 
    Args:
        response: student MIDI dict
        answer:   reference MIDI dict
        params:   optional extra parameters
 
    Returns:
        Result with is_correct and feedback string
    """
    all_correct, feedback = comparison(response, answer)
    return Result(
        is_correct=all_correct,
        feedback_items=[("feedback", "\n".join(feedback))]
    )

In [ ]:
is_correct, feedbacks = comparison(
    response,
    reference
)

print(is_correct)

for feedback in feedbacks:
    print(feedback)

False
Note 1 (with pitch 60) is correct.
Note 2: wrong pitch — expected 62, played 63.
Note 3: difference in start time: 0.15s.
Note 4: difference in duration: 0.20s.
Note 5: wrong pitch — expected 67, played 65.
Note 5: difference in start time: 0.70s.
Note 5: difference in duration: 0.20s.


note5 should be a missing pitch!! Need to check the DTW algorithm!

## Edit Distance

In [ ]:
def compute_cost(note1, note2):
    """
    Cost of aligning (replacing) one note with another, based on pitch.
 
    cost = 0: pitches are identical (a 'match'). 
    cost > 0: different pitches (a 'replacement')
 
    Args:
        note1: dict with keys "pitch" (int), "start" (float), "duration" (float)
        note2: dict with keys "pitch" (int), "start" (float), "duration" (float)
 
    Returns:
        int: cost value >= 0 (lower means more similar pitch)
    """
    return int(abs(note1["pitch"] - note2["pitch"]))


def note_alignment_ED(response_notes, ref_notes, gap_penalty=6):
    """
    Align notes using edit distance (ED). 
    The ED allows for insertions and deletions, which can be useful for 
    evaluating musical practice containing missing/extra notes.
    
    Args:
        response_notes: The student's response MIDI notes to evaluate
        ref_notes: The reference MIDI note
        gap_penalty: cost of leaving a note unaligned (insertion/deletion)
 
    Returns:
        operations: list of transformation ops dicts, in order from first note to last:
            {'type': 'match' or 'replacement' or 'missing' or 'extra', 
            'response_idx': int or None, 
            'ref_idx': int or None, 
            'cost': int}
        D: accumulated cost matrix, shape (N+1, M+1)
    """
    # the rows of D correspond to response notes
    N = len(response_notes)
    # the columns of D correspond to reference notes
    M = len(ref_notes)

    # Build the accumulated cost matrix D of size (N+1 x M+1)
    D = np.zeros((N + 1, M + 1), dtype=int)
    # Boundary conditions: aligning against an empty sequence means every note
    # is unaligned, so the cost is n (or m) times the gap penalty.
    for n in range(1, N + 1):
        D[n, 0] = n * gap_penalty # n extra response notes
    for m in range(1, M + 1):
        D[0, m] = m * gap_penalty # m missing ref notes
    # Recursion (accumulated cost / score matrix D):
    for n in range(1, N + 1):
        for m in range(1, M + 1):
            replace_cost = compute_cost(response_notes[n-1], ref_notes[m-1])
            D[n, m] = min(
                D[n-1, m-1] + replace_cost, # diagonal: match or replacement
                D[n-1, m] + gap_penalty, # vertical: extra note response[n-1]
                D[n, m-1] + gap_penalty, # horizontal: missing response for ref[m-1]
            )

    # Backtrack, classify each transformation ops based on movement direction in D
    operations = []
    n, m = N, M
    while n > 0 or m > 0:
        # boundary conditions: if we are at the most top row or left column, we can only move in one direction
        # at the top row, only horizontal moves possible
        if n == 0: 
            # missing response for ref[m-1] (deletion)
            operations.append({"type": "missing", "response_idx": None,
                            "ref_idx": m - 1, "cost": gap_penalty})
            m -= 1
        # at the most left column, only vertical moves possible
        elif m == 0: 
            # extra note response[n-1] (insertion)
            operations.append({"type": "extra", "response_idx": n - 1,
                            "ref_idx": None, "cost": gap_penalty})
            n -= 1
        # for all other cases, we can move in any direction (diagonal, vertical, horizontal)
        else:
            replace_cost = compute_cost(response_notes[n-1], ref_notes[m-1])
            diag = D[n-1, m-1] + replace_cost # diagonal: match or replacement
            up = D[n-1, m] + gap_penalty # vertical: extra note response[n-1]
            left = D[n, m-1] + gap_penalty # horizontal: missing response for ref[m-1]
            min_cost = min(diag, up, left) # find the minimum cost step
            # classify the transformation ops based on the minimum cost step
            if min_cost == diag: # diagonal -> two notes are aligned (match/replacement)
                operations.append({
                    "type": "match" if replace_cost == 0 else "replacement",
                    "response_idx": n - 1,
                    "ref_idx": m - 1,
                    "cost": replace_cost,
                })
                n, m = n - 1, m - 1
            elif min_cost == up: # vertical -> response[n-1] is extra (insertion)
                operations.append({"type": "extra", "response_idx": n - 1,
                                "ref_idx": None, "cost": gap_penalty})
                n -= 1
            else: # horizontal -> response is missing for ref[m-1] (deletion)
                operations.append({"type": "missing", "response_idx": None,
                                "ref_idx": m - 1, "cost": gap_penalty})
                m -= 1
 
    operations.reverse() # reverse the operations to get them in order from first note to last
    return operations, D